# Submission  | Sebislaw

## Libraries

In [2]:
from os.path  import join
import random
import itertools
import math

import numpy as np
import pandas as pd
from pandas.plotting import scatter_matrix

import matplotlib.pyplot as plt
import plotly.express as px
from pandas.plotting import parallel_coordinates
import seaborn as sns
import ipywidgets as widgets
from IPython.display import display

from sklearn.linear_model import LinearRegression, LassoCV, LogisticRegression, LogisticRegressionCV
from sklearn.model_selection import train_test_split, TimeSeriesSplit, cross_val_score, StratifiedKFold, RandomizedSearchCV
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import brier_score_loss
from sklearn.feature_selection import SelectFromModel
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import mutual_info_classif
from sklearn.neural_network import MLPClassifier

import xgboost as xgb
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
import optuna
# from tabpfn import TabPFNClassifier

## Data

In [3]:
data_path = ''
pd.set_option('display.max_columns', None)

SampleSubmissionStage2 = pd.read_csv(join('..\\..\\..\\data', 'SampleSubmissionStage2.csv'))

MenTest = pd.read_csv(join(data_path, "MenTest.csv"), index_col=0)
MenTrain = pd.read_csv(join(data_path, "MenTrain.csv"), index_col=0)

WomenTest = pd.read_csv(join(data_path, "WomenTest.csv"), index_col=0)
WomenTrain = pd.read_csv(join(data_path, 'WomenTrain.csv'), index_col=0)

# MenTest and WomenTest have the same columns and the same
# 'Season', 'T1_TeamID', 'T1_Score', 'T2_TeamID', 'T2_Score', 'location'
# values in the same order, but Men have only daata for men teams and NaNs in other rows.
# The same for WomenTest.
# Test below fills the NaNs with values from the data frame, where values are present.
Test = MenTest.combine_first(WomenTest)

## Data preparation pipeline

In [4]:
def clear_na_from_x_y(x, y):
    """
    The data frame for final season is in format matching the submission file.
    This function clears NaNs from data.
    """
    # Create masks for training data:
    mask_train = ~np.isnan(x).any(axis=1) & ~np.isnan(y)
    x_clean = x[mask_train]
    y_clean = y[mask_train]
    return x_clean, y_clean

def x_y_from_data_frame(df):
    # Prepare data and labels
    x = df[list(df.columns[6:])].values
    y = np.where(
        df[['T1_Score', 'T2_Score']].isnull().any(axis=1),
        np.nan,
        np.where(df['T1_Score'] - df['T2_Score'] > 0, 1, 0)
    )
    return x, y

## Get data to use models on for testing

In [5]:
THE_LAST_YEAR = 2024
# I will be using train and test from train dataset
x_train_men, y_train_men = x_y_from_data_frame(MenTrain[MenTrain['Season']<THE_LAST_YEAR])
x_train_women, y_train_women = x_y_from_data_frame(WomenTrain[WomenTrain['Season']<THE_LAST_YEAR])
x_test_men, y_test_men = x_y_from_data_frame(MenTrain[MenTrain['Season']==THE_LAST_YEAR])
x_test_women, y_test_women = x_y_from_data_frame(WomenTrain[WomenTrain['Season']==THE_LAST_YEAR])

# Training models

In [6]:
import optuna
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import brier_score_loss
from sklearn.model_selection import StratifiedKFold

# Define an objective function that uses StratifiedKFold CV.
def objective_lr_cv(trial, X, y, n_folds=5):  # Increased folds for better generalization
    # Stronger regularization: Reduce range of C to prevent overfitting
    C = trial.suggest_float("C", 1e-4, 10, log=True)  # Lower upper bound
    
    penalty = trial.suggest_categorical("penalty", ["l1", "l2"])
    solver_all = "liblinear" if penalty == "l1" else "lbfgs"
    solver = solver_all  # More stable for both L1 and L2

    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42)
    scores = []

    for train_idx, val_idx in skf.split(X, y):
        X_train_cv, X_val_cv = X[train_idx], X[val_idx]
        y_train_cv, y_val_cv = y[train_idx], y[val_idx]

        model = Pipeline([
            ("scaler", StandardScaler()),
            ("classifier", LogisticRegression(penalty=penalty, C=C, solver=solver, max_iter=3000))  # Increased max_iter
        ])

        model.fit(X_train_cv, y_train_cv)
        y_pred = model.predict_proba(X_val_cv)[:, 1]
        scores.append(brier_score_loss(y_val_cv, y_pred))

    return np.mean(scores)

###########################
# For all
###########################

x_train_all = np.concatenate((x_train_women, x_train_men))
y_train_all = np.concatenate((y_train_women, y_train_men))

x_test_all = np.concatenate((x_test_women, x_test_men))
y_test_all = np.concatenate((y_test_women, y_test_men))

###########################
study_all = optuna.create_study(direction="minimize")
study_all.optimize(lambda trial: objective_lr_cv(trial, x_train_all, y_train_all), n_trials=100)  # Increased trials for better tuning

best_params_all = study_all.best_params
print("Best Logistic Regression CV Params (all):", best_params_all)

best_lr_all = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(penalty=best_params_all["penalty"],
                                       C=best_params_all["C"],
                                       solver="saga",
                                       max_iter=3000))
])
best_lr_all.fit(x_train_all, y_train_all)
y_pred_all_lr = best_lr_all.predict_proba(x_test_all)[:, 1]
brier_all_lr = brier_score_loss(y_test_all, y_pred_all_lr)
print("Final Logistic Regression Brier Score (all):", brier_all_lr)

[I 2025-03-20 13:22:54,616] A new study created in memory with name: no-name-518179f7-7324-4b35-a05b-f461b2320714
[I 2025-03-20 13:22:54,701] Trial 0 finished with value: 0.16302404932902412 and parameters: {'C': 0.006839645201696326, 'penalty': 'l1'}. Best is trial 0 with value: 0.16302404932902412.
[I 2025-03-20 13:22:54,783] Trial 1 finished with value: 0.15122686178300251 and parameters: {'C': 0.04998822434575589, 'penalty': 'l1'}. Best is trial 1 with value: 0.15122686178300251.
[I 2025-03-20 13:22:55,055] Trial 2 finished with value: 0.1491149883807555 and parameters: {'C': 0.3393617303215821, 'penalty': 'l2'}. Best is trial 2 with value: 0.1491149883807555.
[I 2025-03-20 13:22:55,132] Trial 3 finished with value: 0.17964545539198226 and parameters: {'C': 0.0008450250668689031, 'penalty': 'l2'}. Best is trial 2 with value: 0.1491149883807555.
[I 2025-03-20 13:22:55,247] Trial 4 finished with value: 0.14992687949200165 and parameters: {'C': 0.07139479879667533, 'penalty': 'l1'}. B

[I 2025-03-20 13:23:06,151] Trial 44 finished with value: 0.14881650022371828 and parameters: {'C': 0.287016967113293, 'penalty': 'l1'}. Best is trial 38 with value: 0.14879677082746762.
[I 2025-03-20 13:23:06,258] Trial 45 finished with value: 0.14992264237730013 and parameters: {'C': 0.07145163465159457, 'penalty': 'l1'}. Best is trial 38 with value: 0.14879677082746762.
[I 2025-03-20 13:23:06,485] Trial 46 finished with value: 0.14879998164182243 and parameters: {'C': 0.24627273362651533, 'penalty': 'l1'}. Best is trial 38 with value: 0.14879677082746762.
[I 2025-03-20 13:23:06,560] Trial 47 finished with value: 0.15273363069694593 and parameters: {'C': 0.025678871407295214, 'penalty': 'l1'}. Best is trial 38 with value: 0.14879677082746762.
[I 2025-03-20 13:23:06,643] Trial 48 finished with value: 0.1574580210418747 and parameters: {'C': 0.009885391668603279, 'penalty': 'l1'}. Best is trial 38 with value: 0.14879677082746762.
[I 2025-03-20 13:23:07,147] Trial 49 finished with value

[I 2025-03-20 13:23:18,823] Trial 88 finished with value: 0.14892566732196447 and parameters: {'C': 0.895516898408641, 'penalty': 'l1'}. Best is trial 71 with value: 0.14879643580135354.
[I 2025-03-20 13:23:19,339] Trial 89 finished with value: 0.14884652906767176 and parameters: {'C': 0.4884507843211717, 'penalty': 'l1'}. Best is trial 71 with value: 0.14879643580135354.
[I 2025-03-20 13:23:19,486] Trial 90 finished with value: 0.14922440458834924 and parameters: {'C': 0.10710893647730794, 'penalty': 'l1'}. Best is trial 71 with value: 0.14879643580135354.
[I 2025-03-20 13:23:19,726] Trial 91 finished with value: 0.14879937265004964 and parameters: {'C': 0.23747336854386103, 'penalty': 'l1'}. Best is trial 71 with value: 0.14879643580135354.
[I 2025-03-20 13:23:20,122] Trial 92 finished with value: 0.14882582658503235 and parameters: {'C': 0.3235396371484863, 'penalty': 'l1'}. Best is trial 71 with value: 0.14879643580135354.
[I 2025-03-20 13:23:20,298] Trial 93 finished with value: 0

Best Logistic Regression CV Params (all): {'C': 0.23597973266161204, 'penalty': 'l1'}
Final Logistic Regression Brier Score (all): 0.14742580396326366


## Testing the model

In [7]:
year_range = [i for i in range(2011, 2020)] + [i for i in range(2022, 2025)]
brier_all_list = []

for i in range(len(year_range)):
    
    THE_LAST_YEAR = year_range[i]
    # For data leak validation
#     print(len(MenTrain[MenTrain['Season']<THE_LAST_YEAR]['Season']))
#     print(len(MenTrain[MenTrain['Season']<THE_LAST_YEAR]['Season']))
#     print(len(MenTrain[MenTrain['Season']==THE_LAST_YEAR]))
#     print(MenTrain[MenTrain['Season']<THE_LAST_YEAR]['Season'].head())
#     print(MenTrain[MenTrain['Season']<THE_LAST_YEAR]['Season'].tail())
#     print(MenTrain[MenTrain['Season']==THE_LAST_YEAR]['Season'])
    print("Predictions for season ", year_range[i])
    
    x_train_men, y_train_men = x_y_from_data_frame(MenTrain[MenTrain['Season']<THE_LAST_YEAR])
    x_train_women, y_train_women = x_y_from_data_frame(WomenTrain[WomenTrain['Season']<THE_LAST_YEAR])
    x_test_men, y_test_men = x_y_from_data_frame(MenTrain[MenTrain['Season']==THE_LAST_YEAR])
    x_test_women, y_test_women = x_y_from_data_frame(WomenTrain[WomenTrain['Season']==THE_LAST_YEAR])
    
    x_train_all = np.concatenate((x_train_women, x_train_men))
    y_train_all = np.concatenate((y_train_women, y_train_men))
    x_test_all = np.concatenate((x_test_women, x_test_men))
    y_test_all = np.concatenate((y_test_women, y_test_men))
    
    best_lr_all.fit(x_train_all, y_train_all)
    y_prob_all = best_lr_all.predict_proba(x_test_all)[:, 1]

    score = brier_score_loss(y_test_all, y_prob_all)
    brier_all_list.append(score)
    print('LogisticRegressionCV for All trained on All', score)
    # ----------------------------------------------
print()
print('The mean score when trained on All and tested on All was: ', np.mean(brier_all_list), 
      'with a std of ', np.std(brier_all_list))

# Best params {'C': 0.23059755790690561, 'penalty': 'l1'}

Predictions for season  2011
LogisticRegressionCV for All trained on All 0.15997933731162453
Predictions for season  2012
LogisticRegressionCV for All trained on All 0.1304734705155532
Predictions for season  2013
LogisticRegressionCV for All trained on All 0.16434500881927488
Predictions for season  2014
LogisticRegressionCV for All trained on All 0.14741749597681583
Predictions for season  2015
LogisticRegressionCV for All trained on All 0.12240345146394639
Predictions for season  2016
LogisticRegressionCV for All trained on All 0.15044918729726006
Predictions for season  2017
LogisticRegressionCV for All trained on All 0.13563339864854906
Predictions for season  2018
LogisticRegressionCV for All trained on All 0.15671693670632988
Predictions for season  2019
LogisticRegressionCV for All trained on All 0.1500660439533966
Predictions for season  2022
LogisticRegressionCV for All trained on All 0.14915051865623968
Predictions for season  2023
LogisticRegressionCV for All trained on All

# Create Submission

In [8]:
data_path = ''
pd.set_option('display.max_columns', None)

SampleSubmissionStage2 = pd.read_csv(join('..\\..\\..\\data', 'SampleSubmissionStage2.csv'))

MenTest = pd.read_csv(join(data_path, "MenTest.csv"), index_col=0)
MenTrain = pd.read_csv(join(data_path, "MenTrain.csv"), index_col=0)

WomenTest = pd.read_csv(join(data_path, "WomenTest.csv"), index_col=0)
WomenTrain = pd.read_csv(join(data_path, 'WomenTrain.csv'), index_col=0)

# MenTest and WomenTest have the same columns and the same
# 'Season', 'T1_TeamID', 'T1_Score', 'T2_TeamID', 'T2_Score', 'location'
# values in the same order, but Men have only daata for men teams and NaNs in other rows.
# The same for WomenTest.
# Test below fills the NaNs with values from the data frame, where values are present.
Test = MenTest.combine_first(WomenTest)

In [9]:
x_test, y_test = x_y_from_data_frame(Test)

In [10]:
# I will fill the NaNs (seeds) with 0. It doesn't matter, since those teams won't play in the tournament anyway.
x_test = np.nan_to_num(x_test, nan=0)

In [11]:
predictions = best_lr_all.predict_proba(x_test)[:, 1]

In [12]:
Submission = SampleSubmissionStage2.copy()

In [13]:
Submission['Pred'] = predictions

In [14]:
Submission

,ID,Pred
0,2025_1101_1102,0.840095
1,2025_1101_1103,0.095099
2,2025_1101_1104,0.007057
3,2025_1101_1105,0.972561
4,2025_1101_1106,0.859854
...,...,...
131402,2025_3477_3479,0.345171
131403,2025_3477_3480,0.118576
131404,2025_3478_3479,0.582866
131405,2025_3478_3480,0.262871


In [15]:
Submission.to_csv("LogisticRegressionWithElo_NEW.csv", index=False)

In [16]:
pd.read_csv(join(data_path, "LogisticRegressionWithElo_NEW.csv"))

,ID,Pred
0,2025_1101_1102,0.840095
1,2025_1101_1103,0.095099
2,2025_1101_1104,0.007057
3,2025_1101_1105,0.972561
4,2025_1101_1106,0.859854
...,...,...
131402,2025_3477_3479,0.345171
131403,2025_3477_3480,0.118576
131404,2025_3478_3479,0.582866
131405,2025_3478_3480,0.262871
